In [1]:
import scanpy as sc 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

import scipy.sparse as sp 
import warnings

warnings.filterwarnings("ignore")

import os
import ctypes
import sys

# 1. 先设置 R_HOME
os.environ["R_HOME"] = "/home/pxy/miniconda3/envs/r40/lib/R"

# 2. 【核心黑科技】手动加载 R 的动态库
# 这步操作等同于在终端里设置 LD_LIBRARY_PATH，专门解决 VS Code 找不到库的问题
try:
    # 这是 R 的核心库路径
    libR_path = "/home/pxy/miniconda3/envs/r40/lib/R/lib/libR.so"
    # 强制加载进内存
    ctypes.CDLL(libR_path, mode=ctypes.RTLD_GLOBAL)
    print("✅ 成功强制加载 libR.so")
except OSError as e:
    print(f"❌ 加载失败: {e}")

# 3. 然后再导入其他包
sys.path.append("..") 

import spCLUE
import rpy2.robjects as robjects
print("R 环境路径:", robjects.r['R.home']()[0])

spCLUE.fix_seed(0)

# 定义DLPFC数据集的12个切片ID
slice_ids = [
    "151507", "151508", "151509", "151510",
    "151669", "151670", "151671", "151672",
    "151673", "151674", "151675", "151676"
]

# 用于存储每个切片的ARI结果
ari_results = []

# 数据路径（请根据实际情况确认路径是否正确）
data_dir = '/home/pxy/home/pxy/data/DLPFC/st/'

print(f"Start processing {len(slice_ids)} slices...")

for sample_name in slice_ids:
    print(f"\n{'='*20} Processing Sample: {sample_name} {'='*20}")
    
    # 1. 设置簇的数量 (根据DLPFC数据集的已知Ground Truth)
    # 151669-151672 通常只有5层，其他切片为7层
    if sample_name in ["151669", "151670", "151671", "151672"]:
        n_clusters = 5
    else:
        n_clusters = 7
    
    try:
        # 2. 加载数据
        # 使用 read_visium 加载数据，路径拼接逻辑参考原文件
        adata = sc.read_visium(data_dir + sample_name)
        adata.var_names_make_unique()
        
        # 加载元数据 (Ground Truth)
        meta = pd.read_csv(data_dir + sample_name + "/metadata.tsv", sep="\t")
        meta = meta.set_index("barcode")
        adata.obs["Region"] = meta.loc[adata.obs_names, "layer_guess_reordered"]
        
        # 3. 数据预处理与构图
        # 原文件 Cell 6 的逻辑
        adata = spCLUE.preprocess(adata, n_top_genes=3000)
        adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)
        
        g_spatial = spCLUE.prepare_graph(adata, "spatial", n_neighbors=8)
        g_expr = spCLUE.prepare_graph(adata, "expr", n_neighbors=8)
        graph_dict = {"spatial": g_spatial, "expr": g_expr}
        
        # 4. 模型初始化与训练
        # 原文件 Cell 8 的逻辑
        # 注意：这里将 n_clusters 参数改为动态变量，与当前切片保持一致
        spCLUE_model = spCLUE.spCLUE(
            input_data=adata.obsm['X_pca'],
            hvg_input=adata.obsm['X_pca'],
            graph_dict=graph_dict,
            n_clusters=n_clusters,
            reconstruction_loss='mse',
            gamma=1.0,
            gamma_mask=0.5,
            beta=1.0,
            kappa=5.0,
            # cluster_weight_strategy='min',  # 🔥 置信度策略
            # cluster_warmup_epochs=50,       # 🔥 Warmup
            # dim_input=2000,
            # dim_hidden=256,
            dim_embed=16,
            graph_corr=0.4,
            epochs = 800,
            patience=50,
            min_delta=0.005,
        )
  
        # 训练模型
        _, adata.obsm["spCLUE"] = spCLUE_model.train()
        
        # 5. 聚类
        # 原文件 Cell 10 的逻辑
        refinement = True
        cluster_method = "mclust"
        spCLUE.clustering(
            adata,
            n_clusters,
            key="spCLUE",
            refinement=refinement,
            cluster_methods=cluster_method,
        )
        
        # 6. 计算 ARI
        # 原文件 Cell 12 的逻辑
        # 过滤掉 Ground Truth 为 NA 的区域
        adata_valid = adata[adata.obs.Region.notna()]
        ARI = adjusted_rand_score(adata_valid.obs["Region"], adata_valid.obs["mclust_refined"])
        
        print(f"Sample {sample_name} ARI: {ARI:.4f}")
        ari_results.append(ARI)
        
    except Exception as e:
        print(f"Error processing sample {sample_name}: {e}")

# 7. 输出最终统计结果
print(f"\n{'='*20} Final Results {'='*20}")
if ari_results:
    mean_ari = np.mean(ari_results)
    median_ari = np.median(ari_results)
    print(f"ARI per slice: {[round(x, 4) for x in ari_results]}")
    print(f"Mean ARI: {mean_ari:.4f}")
    print(f"Median ARI: {median_ari:.4f}")
else:
    print("No ARI results collected.")

✅ 成功强制加载 libR.so
R 环境路径: /home/pxy/miniconda3/envs/r40/lib/R
Start processing 12 slices...

==================== Processing Sample: 151507 ====================

预处理单切片数据 (全基因 PCA + 归一化 HVG)

根据原始计数挑选 top 3000 个高变基因...
全量数据归一化 (target_sum=1e4) & Log1p...
  ✓ 已保存归一化后的 HVG 重构目标: (4226, 3000)
将全量稀疏矩阵转换为稠密矩阵以计算 PCA...
计算 PCA (基于全基因归一化数据, n_components=200)...

✅ 预处理完成！
  - adata.X (全量归一化): (4226, 11982)
  - adata.obsm['X_hvg'] (归一化HVG): (4226, 3000)
  - adata.obsm['X_pca'] (全基因PCA): (4226, 200)

构建 spatial 图...
  - 使用空间坐标: (4226, 2)
  ✓ 空间图构建完成:  (4226, 4226), 边数=35800

构建 expr 图...
  - 使用PCA:  (4226, 200)
  ✓ 表达图构建完成: (4226, 4226), 边数=65578
ℹ️ 初始化单切片模型 (CCGCN)
   - 输入:  200维PCA
   - 重构: 2000个HVG
✅ 使用 MSE 重构损失
Training Start =========================>
重构损失类型: MSE
聚类对齐策略: min
Warmup epochs: 50
早停策略: patience=50, min_delta=0.005
损失权重: kappa=5.0, beta=1.0, gamma=1.0


  1%|          | 6/800 [00:01<02:09,  6.15it/s]


Epoch 1:   Loss=12.9728, ARI=0.119
 Cluster=2.5666, Recon=2.5844, MaskRecon=2.5855, LocalAgg=0.6529
  ClusterLoss=2.5629, NegLoss=0.0037


  7%|▋         | 56/800 [00:02<00:17, 42.08it/s]


Epoch 50:   Loss=11.5402, ARI=0.288
 Cluster=2.3942, Recon=2.5695, MaskRecon=2.5565, LocalAgg=0.5298
  ClusterLoss=2.3929, NegLoss=0.0013


...


Epoch 250:   Loss=10.5440, ARI=0.315
 Cluster=1.5254, Recon=2.4698, MaskRecon=2.4776, LocalAgg=0.5310
  ClusterLoss=1.5108, NegLoss=0.0146


 39%|███▊      | 309/800 [00:06<00:10, 48.32it/s]


Epoch 300:   Loss=10.0531, ARI=0.316
 Cluster=1.3054, Recon=2.4631, MaskRecon=2.4348, LocalAgg=0.5067
  ClusterLoss=1.2926, NegLoss=0.0128


 43%|████▎     | 347/800 [00:07<00:09, 48.00it/s]



Early stopping at epoch 348
Best loss: 9.9504
Training Finished =================<
fitting ...
  |======================================================================| 100%
Sample 151676 ARI: 0.2250

==================== Final Results ====================
ARI per slice: [0.1923, 0.2424, 0.3464, 0.2818, 0.2894, 0.3891, 0.4758, 0.4625, 0.3964, 0.3517, 0.2238, 0.225]
Mean ARI: 0.3231
Median ARI: 0.3179
